<a href="https://colab.research.google.com/github/nkcrooks/Movie-Analysis---Python/blob/main/notebooks/00_enrich_trakt_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Enriching Trakt data with API call

## Loading Data

In [ ]:
import pandas as pd
from pathlib import Path
import sys

# Add the project root to sys.path
sys.path.append(str(Path().resolve().parent))

# Set folder path
root_path = Path().resolve().parent
file_path=root_path/"data" / "raw" 
# Get all JSON files
json_files = list(file_path.glob('*.json'))
dfs = {file.stem: pd.read_json(file) for file in json_files}

In [18]:
nkc = pd.DataFrame(dfs['nkc-ratings-movies'])
tmm = pd.DataFrame(dfs['taylor-ratings-movies'])
print(nkc.shape)
print(tmm.shape)

(299, 4)
(136, 4)


In [19]:
nkc['viewer'] = 'nowell'
tmm['viewer'] = 'taylor'

df=pd.concat([nkc, tmm]).reset_index(drop=True)
df.shape
display(df.head(), df.tail())

,rated_at,rating,type,movie,viewer
0,2025-07-01 13:56:58+00:00,6,movie,"{'title': 'Hercules', 'year': 1997, 'ids': {'t...",nowell
1,2025-07-01 11:56:19+00:00,7,movie,"{'title': 'KPop Demon Hunters', 'year': 2025, ...",nowell
2,2025-06-28 18:52:35+00:00,6,movie,"{'title': 'Big Man', 'year': 2025, 'ids': {'tr...",nowell
3,2025-06-28 18:27:50+00:00,6,movie,"{'title': 'Two Distant Strangers', 'year': 202...",nowell
4,2025-06-15 22:10:22+00:00,3,movie,"{'title': 'Force Majeure', 'year': 2014, 'ids'...",nowell


,rated_at,rating,type,movie,viewer
430,2021-04-09 00:00:00+00:00,8,movie,"{'title': 'The Truman Show', 'year': 1998, 'id...",taylor
431,2021-04-05 00:00:00+00:00,5,movie,"{'title': 'The Weekend', 'year': 2016, 'ids': ...",taylor
432,2021-04-04 00:00:00+00:00,7,movie,"{'title': 'The Karate Kid', 'year': 1984, 'ids...",taylor
433,2021-04-04 00:00:00+00:00,8,movie,"{'title': 'Django Unchained', 'year': 2012, 'i...",taylor
434,2021-04-04 00:00:00+00:00,5,movie,"{'title': 'Someone Great', 'year': 2019, 'ids'...",taylor



## Data Cleaning


### Column structure

In [20]:
type(df['movie'])

pandas.core.series.Series

In [21]:
print(type(nkc['movie'].iloc[1]))
print(nkc['movie'].iloc[1]['title'])
nkc['movie'].at[1]

<class 'dict'>
KPop Demon Hunters


{'title': 'KPop Demon Hunters',
 'year': 2025,
 'ids': {'trakt': 638383,
  'slug': 'kpop-demon-hunters-2025',
  'imdb': 'tt14205554',
  'tmdb': 803796}}

In [22]:
expanded = pd.json_normalize(df['movie'])
result = pd.concat([df.drop(columns='movie'), expanded], axis=1)
result.sort_values(by='title', ascending=False)
df=result.copy()

### Column Structure

In [23]:
df.columns

Index(['rated_at', 'rating', 'type', 'viewer', 'title', 'year', 'ids.trakt',
       'ids.slug', 'ids.imdb', 'ids.tmdb'],
      dtype='object')

In [24]:
cols = df.columns

# ['rated_at', 'rating', 'type', 'viewer', 'title', 'year', 'ids.trakt','ids.slug', 'ids.imdb', 'ids.tmdb']

first = ['title', 'rating', 'year', 'viewer', 'rated_at']

remaining = [col for col in cols if col not in first]
cols = first + remaining

df = df[cols]

df.head()

,title,rating,year,viewer,rated_at,type,ids.trakt,ids.slug,ids.imdb,ids.tmdb
0,Hercules,6,1997.0,nowell,2025-07-01 13:56:58+00:00,movie,6983,hercules-1997,tt0119282,11970
1,KPop Demon Hunters,7,2025.0,nowell,2025-07-01 11:56:19+00:00,movie,638383,kpop-demon-hunters-2025,tt14205554,803796
2,Big Man,6,2025.0,nowell,2025-06-28 18:52:35+00:00,movie,1231985,big-man-2025,tt37244927,1502087
3,Two Distant Strangers,6,2020.0,nowell,2025-06-28 18:27:50+00:00,movie,623844,two-distant-strangers-2020,tt13472984,787428
4,Force Majeure,3,2014.0,nowell,2025-06-15 22:10:22+00:00,movie,163864,force-majeure-2014,tt2121382,265189


### Data Describe

In [25]:
display(df.dtypes)
display(df.head())
display(df.info())

title                     object
rating                     int64
year                     float64
viewer                    object
rated_at     datetime64[ns, UTC]
type                      object
ids.trakt                  int64
ids.slug                  object
ids.imdb                  object
ids.tmdb                   int64
dtype: object

,title,rating,year,viewer,rated_at,type,ids.trakt,ids.slug,ids.imdb,ids.tmdb
0,Hercules,6,1997.0,nowell,2025-07-01 13:56:58+00:00,movie,6983,hercules-1997,tt0119282,11970
1,KPop Demon Hunters,7,2025.0,nowell,2025-07-01 11:56:19+00:00,movie,638383,kpop-demon-hunters-2025,tt14205554,803796
2,Big Man,6,2025.0,nowell,2025-06-28 18:52:35+00:00,movie,1231985,big-man-2025,tt37244927,1502087
3,Two Distant Strangers,6,2020.0,nowell,2025-06-28 18:27:50+00:00,movie,623844,two-distant-strangers-2020,tt13472984,787428
4,Force Majeure,3,2014.0,nowell,2025-06-15 22:10:22+00:00,movie,163864,force-majeure-2014,tt2121382,265189


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 435 entries, 0 to 434
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype              
---  ------     --------------  -----              
 0   title      435 non-null    object             
 1   rating     435 non-null    int64              
 2   year       431 non-null    float64            
 3   viewer     435 non-null    object             
 4   rated_at   435 non-null    datetime64[ns, UTC]
 5   type       435 non-null    object             
 6   ids.trakt  435 non-null    int64              
 7   ids.slug   435 non-null    object             
 8   ids.imdb   435 non-null    object             
 9   ids.tmdb   435 non-null    int64              
dtypes: datetime64[ns, UTC](1), float64(1), int64(3), object(5)
memory usage: 34.1+ KB


None

In [26]:
df.isnull().sum()

title        0
rating       0
year         4
viewer       0
rated_at     0
type         0
ids.trakt    0
ids.slug     0
ids.imdb     0
ids.tmdb     0
dtype: int64

In [27]:
duplicateRows = df[df.duplicated()]
duplicateRows

,title,rating,year,viewer,rated_at,type,ids.trakt,ids.slug,ids.imdb,ids.tmdb


## Data Enrichment

Enrich data with API call

In [ ]:
import requests
import json
import time
import random
from requests.adapters import HTTPAdapter, Retry

In [29]:
# Unique IDs to avoid duplicate API calls
unique_imdb_ids = df['ids.imdb'].unique()

In [ ]:
from constants import OMDB_API_KEY

# Access API
OMDB_API_KEY
OMDB_URL = 'http://www.omdbapi.com/'

In [ ]:
# Setup retry session
session = requests.Session()
retries = Retry(total=5, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504])
session.mount('http://', HTTPAdapter(max_retries=retries))

def fetch_movie_details(imdb_id):
    if pd.isnull(imdb_id):
        return None

    params = {'apikey': OMDB_API_KEY, 'i': imdb_id, 'plot': 'short'}
    try:
        response = session.get(OMDB_URL, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('Response') == 'True':
                # Keep only the desired fields
                filtered_data = {
                    'IMDb Id': imdb_id,
                    'Title': data.get('Title'),
                    'URL': f"https://www.imdb.com/title/{imdb_id}/",
                    'Title Type': data.get('Type'),
                    'IMDb Rating': data.get('imdbRating'),
                    'Runtime (mins)': data.get('Runtime', 'N/A').replace(' min', ''),
                    'Release Year': data.get('Year'),
                    'Genres': data.get('Genre'),
                    'Num Votes': data.get('imdbVotes'),
                    'Release Date': data.get('Released'),
                    'Directors': data.get('Director'),
                    'Actors': data.get('Actors'),
                    'Language': data.get('Language'),
                    'Country': data.get('Country'),
                    'Awards': data.get('Awards'),
                    'Metascore': data.get('Metascore'),
                    'Box Office': data.get('BoxOffice'),
                }
                return filtered_data
        return None
    except Exception as e:
        print(f"Error fetching {imdb_id}: {e}")
        return None

In [31]:
# Load or create progress file
try:
    enriched_df = pd.read_csv('enriched_data.csv')
    completed_ids = set(enriched_df['ids.imdb'])
    print(f"Resuming from {len(completed_ids)} records already fetched.")
except FileNotFoundError:
    enriched_df = pd.DataFrame()
    completed_ids = set()

# Prepare list of IDs to process
to_fetch = df[~df['ids.imdb'].isin(completed_ids) & df['ids.imdb'].notna()]

# Loop and fetch data
batch_size = 50
batch_results = []

for idx, row in to_fetch.iterrows():
    imdb_id = row['ids.imdb']

    movie_data = fetch_movie_details(imdb_id)
    if movie_data:
        # Add source row info to movie_data for easier merging later
        movie_data['ids.imdb'] = imdb_id
        batch_results.append(movie_data)

    if len(batch_results) >= batch_size:
        batch_df = pd.DataFrame(batch_results)
        enriched_df = pd.concat([enriched_df, batch_df], ignore_index=True)
        enriched_df.to_csv('enriched_data.csv', index=False)
        print(f"Saved {len(enriched_df)} total records...")
        batch_results = []  # Clear batch
        time.sleep(1)  # Pause between batches

# Final save
if batch_results:
    batch_df = pd.DataFrame(batch_results)
    enriched_df = pd.concat([enriched_df, batch_df], ignore_index=True)
    enriched_df.to_csv('enriched_data.csv', index=False)
    print(f"Final save: {len(enriched_df)} total records.")

print("Enrichment complete ✅")

Saved 50 total records...
Saved 100 total records...
Saved 150 total records...
Saved 200 total records...
Saved 250 total records...
Saved 300 total records...
Saved 350 total records...
Saved 400 total records...
Final save: 435 total records.
Enrichment complete ✅


## Final Validation

In [32]:
print(df.shape)
print(enriched_df.shape)

(435, 10)
(435, 18)


In [33]:
enriched_df.isna().sum()

IMDb Id            0
Title              0
URL                0
Title Type         0
IMDb Rating        0
Runtime (mins)     0
Release Year       0
Genres             0
Num Votes          0
Release Date       0
Directors          0
Actors             0
Language           0
Country            0
Awards             0
Metascore          0
Box Office        11
ids.imdb           0
dtype: int64

In [34]:
enriched_df['Title Type'].value_counts()
rowstodrop=enriched_df.loc[enriched_df['Title Type']=='series']
enriched_df=enriched_df.drop(rowstodrop.index)

In [35]:
from typing_extensions import final
# Merge back the viewer and date rated
final_df = pd.merge(
    df[['ids.imdb', 'viewer', 'rated_at', 'rating']],  # Original ratings with viewer info
    enriched_df,  # Enriched movie info
    on='ids.imdb',
    how='right'
)

# Rename for clarity
final_df = final_df.rename(columns={'rated_at':'Date Rated'})

# Reorder columns
final_df = final_df[[
    'ids.imdb', 'viewer', 'rating', 'Date Rated', 'Title', 'URL', 'Title Type',
    'IMDb Rating', 'Runtime (mins)', 'Release Year', 'Genres',
    'Num Votes', 'Release Date', 'Directors', 'Actors',
    'Language', 'Country', 'Awards',
       'Metascore', 'Box Office']]

final_df.drop_duplicates(inplace=True)

In [36]:
display(final_df.head())
print(final_df.loc[final_df['viewer'] == 'taylor'].shape)
print(final_df.loc[final_df['viewer'] == 'nowell'].shape)

,ids.imdb,viewer,rating,Date Rated,Title,URL,Title Type,IMDb Rating,Runtime (mins),Release Year,Genres,Num Votes,Release Date,Directors,Actors,Language,Country,Awards,Metascore,Box Office
0,tt0119282,nowell,6,2025-07-01 13:56:58+00:00,Hercules,https://www.imdb.com/title/tt0119282/,movie,7.3,93,1997,"Animation, Action, Adventure","272,825",27 Jun 1997,"Ron Clements, John Musker","Tate Donovan, Susan Egan, James Woods","English, Spanish, Greek",United States,Nominated for 1 Oscar. 9 wins & 16 nominations...,74,"$99,112,101"
1,tt14205554,nowell,7,2025-07-01 11:56:19+00:00,KPop Demon Hunters,https://www.imdb.com/title/tt14205554/,movie,7.6,95,2025,"Animation, Action, Adventure","85,772",20 Jun 2025,"Chris Appelhans, Maggie Kang","Arden Cho, May Hong, Ji-young Yoo","English, Korean","United States, Canada",2 nominations total,77,"$18,000,000"
2,tt37244927,nowell,6,2025-06-28 18:52:35+00:00,Big Man,https://www.imdb.com/title/tt37244927/,movie,5.8,N/A,2025,"Short, Drama, Music",117,18 Jun 2025,Aneil Karia,"Stormzy, Klevis Brahja, Jaydon Eastman",English,United Kingdom,N/A,N/A,N/A
3,tt13472984,nowell,6,2025-06-28 18:27:50+00:00,Two Distant Strangers,https://www.imdb.com/title/tt13472984/,movie,6.9,32,2020,"Short, Drama, Sci-Fi","21,920",20 Nov 2020,"Travon Free, Martin Desmond Roe","Joey Bada$$, Zaria, Vincent Mordente",English,United States,Won 1 Oscar. 2 wins total,N/A,N/A
4,tt2121382,nowell,3,2025-06-15 22:10:22+00:00,Force Majeure,https://www.imdb.com/title/tt2121382/,movie,7.2,120,2014,"Comedy, Drama","69,765",30 Dec 2014,Ruben Östlund,"Johannes Kuhnke, Lisa Loven Kongsli, Clara Wet...","Swedish, Norwegian, English, French, Italian","Sweden, France, Norway, Denmark, Italy",Nominated for 1 BAFTA Award31 wins & 41 nomina...,87,"$1,359,497"


(133, 20)
(291, 20)


In [40]:
# Save final result

# final_df.to_csv('enriched_movies.csv', index=False)
final_df.to_csv('../data/processed/enriched_movies.csv', index=False)
print("Final merged dataset saved ✅")

Final merged dataset saved ✅
